# 🎙️ Sesame Clone — a conversational voice assistant on a free Colab GPU

**Whisper (ears) → Gemma (brain) → Sesame CSM-1B (voice)**

This notebook reproduces the *architecture* behind Sesame's demo — the thing that makes it
sound natural — not just "text in, audio out". Everything runs on a free Colab T4.

The brain runs **either** on the Google AI **Gemma API** (default — nothing on your GPU, much
bigger model) **or** locally in 4-bit. One flag, `LLM_BACKEND`, switches between them. The voice
model always runs locally: there is no hosted CSM API, and it's the part that actually needs a GPU.

---

## What Sesame's research actually says

From [*Crossing the uncanny valley of conversational voice*](https://www.sesame.com/research/crossing_the_uncanny_valley_of_voice)
(Schalkwyk, Kumar, Lyth, Eskimez, Hodari, Resnick, Sanabria, Jiang et al.):

**The problem they name is "one-to-many".** One sentence can be spoken in a thousand valid ways.
Text alone doesn't tell you which one. Classic TTS has no mechanism to choose, so it hedges and
picks the safe, flat, neutral reading every time. That flatness *is* the uncanny valley.

**Their fix: condition on the conversation, including its audio.** CSM (Conversational Speech
Model) is a single-stage multimodal transformer that takes interleaved **text *and* audio** history
as context and generates RVQ audio codes directly. Prosody stops being guessed from a lone
sentence and starts being inferred from how the conversation has actually been going.

**Architecture — two autoregressive transformers over a split-RVQ tokenizer:**

| Component | Job |
|---|---|
| **Backbone** (Llama-style, 1B in CSM-1B) | Reads interleaved text + audio, predicts **codebook 0** — the *semantic* token |
| **Depth / audio decoder** (~100M, 4 layers) | Predicts the remaining **31 acoustic codebooks** for that frame |
| **Mimi codec** (Kyutai) | 24 kHz audio ⇄ discrete tokens at **12.5 Hz** (one frame per 80 ms) |

Splitting it this way keeps the big model on meaning and a tiny model on timbre — that's what
makes it fast enough to hold a conversation.

**Compute amortization (their training trick).** The audio decoder would need an effective batch of
`B × S × N` (batch × sequence × 32 codebooks) — brutal memory. So they train the decoder on a
random **1/16 subset of frames** while codebook 0 trains on *every* frame. No perceptible quality
loss. This notebook exposes that exact knob in the fine-tuning section
(`depth_decoder_labels_ratio=1/16`).

**Scale.** Trained on ~**1M hours** of mostly-English audio, in three sizes: Tiny (1B backbone /
100M decoder — the open one), Small (3B / 250M), Medium (8B / 300M). Bigger = more contextually
appropriate, not just cleaner.

**How they measured it.** Word-error-rate and speaker similarity were already saturated, so they
invented two context-sensitive tests: **homograph disambiguation** (does it say "lead" the metal or
"lead" the verb?) and **pronunciation continuation consistency** (does a name keep the same
pronunciation across turns?). Subjectively, with CMOS studies on Expresso: on *isolated* sentences
CSM is near parity with human recordings — but **once conversational context is included, humans
still clearly prefer real humans.** They published the gap rather than hiding it.

**What they said is still missing:** full duplex — real turn-taking, interruption, and pacing.
CSM generates speech; it doesn't yet decide *when* to speak.

---

## What that means for the code below

Four design rules, taken straight from the paper. These are the difference between this and any
generic TTS demo:

1. **The context buffer stores audio, not just text.** Every turn — yours *and* the assistant's —
   is kept as `(speaker_id, text, waveform)` and fed back into CSM. This is the whole ballgame.
2. **A pinned voice anchor.** CSM-1B is a *base* model with no fixed voice; with an empty context it
   invents a new speaker every call. We bootstrap one reference utterance and pin it to the front of
   the context forever.
3. **A real frame budget.** The backbone tops out at 2048 positions and audio burns 12.5 of them per
   second. We count text tokens and audio frames exactly, and evict oldest-first while never
   evicting the anchor.
4. **Sentence-level chunking.** Speak sentence by sentence, feeding each finished chunk back as
   context. First audio arrives sooner and prosody still carries across the sentence boundary.

> ⚠️ **Responsible use.** Sesame's license explicitly prohibits impersonating real people without
> consent, deception, and fraud. Note also that the original Sesame repo watermarks its output with
> `silentcipher`; the 🤗 Transformers integration used here does **not**. If you ship anything from
> this, add watermarking back. Details in the last cell.

## How this actually runs in Colab

Worth understanding before you hit run, because a few of these will surprise you.

**Colab gives you a disposable Linux VM with a GPU.** Your browser is just a terminal into it. The
Python kernel lives on Google's machine; `nvidia-smi` in Step 0 is talking to a T4 in a datacenter,
not your laptop. Closing the tab doesn't stop the kernel — but the VM **is deleted** when the
session ends (~12 h max, or ~90 min idle), taking every downloaded weight and generated `.wav`
with it. Anything you want to keep, download or write to Drive.

**Weights re-download every fresh session** — ~5 GB for CSM, ~1.6 GB for Whisper, a few minutes on
Colab's fast link. They land in `~/.cache/huggingface` on the ephemeral disk. Mount Drive and set
`HF_HOME` there if you're iterating a lot and hate waiting.

**The microphone is the fiddly part.** Python is on Google's VM; your mic is on your desk. There is
no way for server-side Python to open it. So `record_mic()` does a round trip:

```
browser: MediaRecorder captures WebM/Opus  ->  base64 data URL
   |  (google.colab.output.eval_js bridge)
Python on the VM: decode -> ffmpeg -> 24 kHz mono -> numpy
   |
Whisper -> Gemma -> CSM  ->  numpy waveform
   |  (IPython.display.Audio embeds it as base64)
browser: <audio> element plays it back
```

Practical consequences: **Chrome or Edge on desktop** (Safari and mobile browsers are unreliable
here), you must click **Allow** on the mic prompt, and recording is **fixed-length** — it grabs
exactly `RECORD_SECONDS` and stops. There's no barge-in and no voice-activity detection, which is
the half-duplex limitation Sesame themselves call out as unsolved. Step 13 lets you type instead if
the mic won't cooperate.

**Latency, end to end.** Roughly what to expect per turn on a free T4:

| Stage | `"api"` | `"local"` |
|---|---|---|
| Record | fixed (you choose) | fixed |
| Whisper transcribe | 0.4–1.0 s | 0.3–0.8 s |
| Gemma reply | 0.4–1.2 s (network) | 0.6–1.5 s |
| **CSM first chunk** | **1.5–3 s** | **1.5–3 s** |
| → first audio | **~3–5 s** | **~3–5 s** |

CSM dominates either way, which is exactly why the LLM is the piece worth moving off the GPU — you
give up nothing on the critical path. Sesame's demo hits sub-second because it streams audio out as
frames are generated and runs on far better hardware than a T4; sentence chunking is our
approximation of the first half of that.

---

## Step 0 — Get a GPU

`Runtime → Change runtime type → T4 GPU`, then run this.

Yes, you still need a GPU even on the API path — CSM runs locally regardless.

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "!! No GPU. Runtime > Change runtime type > T4 GPU")

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > T4 GPU"

MAJOR, MINOR = torch.cuda.get_device_capability()
SUPPORTS_BF16 = MAJOR >= 8
print(f"torch {torch.__version__} | compute capability {MAJOR}.{MINOR} | bf16 supported: {SUPPORTS_BF16}")
if not SUPPORTS_BF16:
    print("\nT4 detected (sm_75): no native bfloat16. CSM will run in float32 (~5 GB, safe).")
    print("fp16 is ~4x faster but CSM's RVQ sampling can go unstable in it -- there's a knob below.")

## Step 1 — Install

CSM needs `transformers >= 4.52.1`. **If pip upgrades anything, restart the runtime**
(`Runtime → Restart session`) and then continue from the *next* cell — you don't need to re-run this one.

In [ ]:
%pip install -q -U "transformers>=4.56" accelerate bitsandbytes peft soundfile librosa datasets google-genai num2words
%pip install -q silentcipher   # audio watermarking -- see the note in Step 7c

import transformers
print("transformers", transformers.__version__)
try:
    import google.genai as _g; print("google-genai", _g.__version__)
except Exception as e:
    print("google-genai import failed (restart the runtime):", e)
print("\nIf that upgraded packages: Runtime > Restart session, then continue from the next cell.")

## Step 2 — Credentials

Two keys, and the Colab **secrets** panel (🔑 in the left sidebar) is the right place for both —
add them once, toggle "Notebook access", and this cell is fully automatic forever. Never paste a
key into a cell you might share.

| Secret | Where to get it | Needed for |
|---|---|---|
| `HF_TOKEN` | [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) | Downloading CSM-1B |
| `GOOGLE_API_KEY` | [aistudio.google.com/apikey](https://aistudio.google.com/apikey) | The Gemma API (free tier, no card) |

**CSM-1B is gated.** Open 🔗 [`sesame/csm-1b`](https://huggingface.co/sesame/csm-1b) and click
"Agree and access repository" (instant) *before* running this. Forgetting is the #1 cause of a
`401`/`403` later.

*Only if you set `LLM_BACKEND = "local"`:* also accept
🔗 [`google/gemma-3-4b-it`](https://huggingface.co/google/gemma-3-4b-it). On the API path you don't
need it — the Gemma weights never touch this machine.

In [ ]:
import os
from huggingface_hub import login, whoami

def _secret(name):
    """Colab secrets first, then env vars."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name)

# ---- Hugging Face (always needed: CSM-1B lives there) ----------------------
hf_token = _secret("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()  # interactive prompt
print("🤗 Logged in as:", whoami()["name"])

# ---- Google AI Studio (needed for LLM_BACKEND = "api") ---------------------
GOOGLE_API_KEY = _secret("GOOGLE_API_KEY") or _secret("GEMINI_API_KEY")
if GOOGLE_API_KEY:
    print("🔑 Google API key found.")
else:
    import getpass
    GOOGLE_API_KEY = getpass.getpass("Google AI Studio key (blank to skip and run Gemma locally): ").strip() or None
    print("🔑 Google API key:", "set" if GOOGLE_API_KEY else "not set -- use LLM_BACKEND = 'local'")

## Step 3 — Configuration

### Where each model runs

| | Runs on | Why |
|---|---|---|
| **Whisper** (ears) | Colab GPU | Small, fast, and audio uploads cost more latency than they save |
| **Gemma** (brain) | **Your choice** | Text in, text out — trivially remotable |
| **CSM-1B** (voice) | Colab GPU, always | No hosted CSM API exists. This is what the GPU is *for* |

### `LLM_BACKEND = "api"` (default) vs `"local"`

| | `"api"` | `"local"` |
|---|---|---|
| Model you get | Up to **31B** | 4B in 4-bit |
| VRAM for the LLM | **0 GB** | ~3.2 GB |
| Extra latency/turn | ~0.4–1.2 s network | none |
| Works offline | ❌ needs a key | ✅ |
| Rate limits | ~15 req/min free tier | none |

Both leave CSM untouched, so voice quality is identical — the choice only affects *what it says*
and how much VRAM is left over.

### Memory budget on a 15 GB T4

| | `"api"` | `"local"` |
|---|---|---|
| CSM-1B + Mimi (fp32) | 5.6 GB | 5.6 GB |
| Gemma | — | 3.2 GB |
| Whisper | 1.6 GB (`large-v3-turbo`) | 0.5 GB (`small`) |
| KV cache + activations | ~2 GB | ~2 GB |
| **Total** | **~9 GB** ✅ | **~11 GB** ✅ |

Moving Gemma off the GPU buys ~3 GB, and the best thing to spend it on is a **better Whisper** —
transcription errors poison everything downstream, so `large-v3-turbo` is the single biggest
quality win available here.

In [ ]:
import torch

# ---- where the brain runs ---------------------------------------------------
LLM_BACKEND = "api"        # "api"   -> Google AI Studio (Gemma API), 0 GB VRAM
                           # "local" -> Gemma on this GPU in 4-bit, no network

# ---- models -----------------------------------------------------------------
CSM_MODEL_ID = "sesame/csm-1b"

# API path: "auto" picks the best Gemma your key can see (listed in the next cell).
# Pin one explicitly if you prefer, e.g. "gemma-3-27b-it" or "gemma-4-31b-it".
GEMMA_API_MODEL = "auto"
GEMMA_API_PREFERENCE = [       # first available wins
    "gemma-3-27b-it",          # you asked for Gemma 3 -- this is the biggest of them
    "gemma-3-12b-it",
    "gemma-4-31b-it",          # Gemma 4 (Apr 2026) is a drop-in upgrade if you want it
    "gemma-4-12b-it",
    "gemma-3-4b-it",
]

# Local path only:
LLM_MODEL_ID = "google/gemma-3-4b-it"     # low-memory fallback: "google/gemma-3-1b-it"
LOAD_LLM_IN_4BIT = True                   # set False only if you picked the 1b model

# Whisper: the API path frees ~3 GB, so spend it on better transcription
ASR_MODEL_ID = ("openai/whisper-large-v3-turbo" if LLM_BACKEND == "api"
                else "openai/whisper-small")

assert LLM_BACKEND in ("api", "local")
if LLM_BACKEND == "api" and not GOOGLE_API_KEY:
    raise ValueError("LLM_BACKEND='api' needs GOOGLE_API_KEY (Step 2), or set LLM_BACKEND='local'.")

# ---- precision --------------------------------------------------------------
# T4 (sm_75) has no bfloat16. float32 is the safe default; fp16 is ~4x faster but
# CSM's RVQ sampling can produce static/garbled audio in it. Try it, keep it if it sounds fine.
CSM_USE_FP16_ON_T4 = False

if SUPPORTS_BF16:
    CSM_DTYPE = torch.bfloat16
else:
    CSM_DTYPE = torch.float16 if CSM_USE_FP16_ON_T4 else torch.float32
LLM_COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
DEVICE = "cuda"

# ---- CSM / Mimi constants (from the paper + config) -------------------------
SAMPLE_RATE   = 24_000    # Mimi operates at 24 kHz
FRAME_RATE    = 12.5      # 12.5 Hz -> one audio frame per 80 ms
NUM_CODEBOOKS = 32        # 1 semantic (backbone) + 31 acoustic (depth decoder)
CSM_MAX_SEQ   = 2048      # backbone max_position_embeddings

# Quality window, separate from the hard sequence limit above. Every context segment costs
# prefill time; past a few turns latency grows and quality does not. 2-4 is the sweet spot.
MAX_CONTEXT_TURNS = 4

# torch.compile the CSM decoder. Reported as the difference between sub-realtime and
# real-time on consumer GPUs -- but warmup costs 1-3 min and varying context lengths can
# trigger recompiles. Leave off while iterating; turn on for a long session.
COMPILE_CSM = False

# ---- speaker ids ------------------------------------------------------------
ASSISTANT = 0
USER      = 1

# ---- transformers 4.x/5.x compatibility -------------------------------------
import transformers
DTYPE_KEY = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"

print(f"brain    : {LLM_BACKEND.upper()} "
      f"({GEMMA_API_MODEL if LLM_BACKEND == 'api' else LLM_MODEL_ID})")
print(f"ears     : {ASR_MODEL_ID}")
print(f"voice    : {CSM_MODEL_ID} in {CSM_DTYPE}")
print(f"compat   : {DTYPE_KEY}=")

## Step 4 — Load the three models

First run downloads ~10 GB, so give it a few minutes.

In [ ]:
import gc, time, numpy as np, torch
from transformers import (AutoProcessor, CsmForConditionalGeneration, AutoTokenizer,
                          AutoModelForCausalLM, BitsAndBytesConfig, pipeline)

def vram(tag=""):
    print(f"   VRAM {torch.cuda.memory_allocated()/1e9:5.2f} GB allocated  {tag}")

# ---------------------------------------------------------------- CSM (voice)
print("Loading CSM-1B ...")
t0 = time.time()
csm_processor = AutoProcessor.from_pretrained(CSM_MODEL_ID)
csm_model = CsmForConditionalGeneration.from_pretrained(
    CSM_MODEL_ID, **{DTYPE_KEY: CSM_DTYPE}
).to(DEVICE).eval()
print(f"   done in {time.time()-t0:.0f}s"); vram("after CSM")

# ---------------------------------------------------------------- Gemma (brain)
llm_model = llm_tokenizer = None
if LLM_BACKEND == "local":
    print(f"Loading {LLM_MODEL_ID} ...")
    t0 = time.time()
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)

    llm_kwargs = {"device_map": "auto", DTYPE_KEY: LLM_COMPUTE_DTYPE}
    if LOAD_LLM_IN_4BIT:
        llm_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=LLM_COMPUTE_DTYPE,
        )

    try:
        llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, **llm_kwargs)
    except (ValueError, KeyError):
        # gemma-3-4b+ are multimodal checkpoints (Gemma3ForConditionalGeneration)
        from transformers import AutoModelForImageTextToText
        llm_model = AutoModelForImageTextToText.from_pretrained(LLM_MODEL_ID, **llm_kwargs)
    llm_model.eval()
    print(f"   done in {time.time()-t0:.0f}s"); vram("after Gemma")
else:
    print("Gemma runs via the API -- nothing loaded onto the GPU.")

# ---------------------------------------------------------------- Whisper (ears)
print(f"Loading {ASR_MODEL_ID} ...")
t0 = time.time()
asr = pipeline("automatic-speech-recognition", model=ASR_MODEL_ID,
               device=0, **{DTYPE_KEY: torch.float16})
print(f"   done in {time.time()-t0:.0f}s"); vram("after Whisper")

gc.collect(); torch.cuda.empty_cache()
print("\nLocal models are up.")

### Step 4a — (Optional) Compile the decoder

> *"`torch.compile` the decoder — reported as the difference between sub-realtime and real-time on
> consumer GPUs."*

The guide also frames this as how you **pay for the watermark**: keep the transparency mechanism,
buy the performance back here rather than by stripping it out.

The catch on Colab is warmup — 1–3 minutes on the first call, and varying context lengths can
trigger recompiles that cost more than they save. Leave `COMPILE_CSM = False` while you're
iterating; switch it on for a long session where the warmup amortizes.

Real-time factor is the number to watch. **Below 1.0× is a hard requirement** — at 0.5× you fall
behind and stutter. `speak()` prints it after every turn and warns if it goes over.

In [ ]:
if COMPILE_CSM:
    print("Enabling static cache + CUDA graphs (first generation will be slow) ...")
    csm_model.generation_config.max_length = 1024     # big enough to avoid recompiles
    csm_model.generation_config.max_new_tokens = None
    csm_model.generation_config.cache_implementation = "static"
    csm_model.depth_decoder.generation_config.cache_implementation = "static"
    print("   done -- expect 1-3 min on the first speak() call, fast after.")
else:
    print("torch.compile off. Set COMPILE_CSM = True before this cell for a long session.")

### Step 4b — Connect to the Gemma API

Rather than hardcoding a model ID that Google may rename, this **asks your key what it can see** and
picks the best match. Whatever it prints is what you're actually talking to.

**One real gotcha, and it will bite you if you skip it:** Gemma models on the Gemini API reject the
`system_instruction` parameter outright — you get `400 Developer instruction is not enabled for
models/gemma-...`. Gemini models accept it; Gemma models don't. The workaround is to fold the system
prompt into a user turn instead, which the next section does.

Skip this cell entirely if `LLM_BACKEND = "local"`.

In [ ]:
genai_client = None

if LLM_BACKEND == "api":
    from google import genai
    from google.genai import types

    genai_client = genai.Client(api_key=GOOGLE_API_KEY)

    # what can this key actually reach?
    available = []
    for m in genai_client.models.list():
        name = m.name.split("/")[-1]
        if "gemma" in name.lower():
            available.append(name)

    print("Gemma models visible to your key:")
    for n in sorted(available):
        print("   -", n)

    if GEMMA_API_MODEL == "auto":
        pick = next((p for p in GEMMA_API_PREFERENCE if p in available), None)
        if pick is None:
            pick = sorted(available)[-1] if available else None
        if pick is None:
            raise RuntimeError(
                "No Gemma models available to this key. Check it at aistudio.google.com/apikey, "
                "or set LLM_BACKEND = 'local'."
            )
        GEMMA_API_MODEL = pick

    print(f"\n-> using: {GEMMA_API_MODEL}")

    # smoke test (also proves the key works before we build the whole pipeline)
    r = genai_client.models.generate_content(
        model=GEMMA_API_MODEL,
        contents="Say hello in exactly four words.",
        config=types.GenerateContentConfig(max_output_tokens=32, temperature=0.7),
    )
    print("   test reply:", (r.text or "").strip())
else:
    print("Local backend -- skipping API setup.")

## Step 5 — Audio plumbing

Colab can't touch your microphone from Python, so we go through a bit of JavaScript
(`MediaRecorder`) and convert the resulting WebM/Opus to 24 kHz mono with `ffmpeg`.

**Sample rates are asymmetric on purpose.** Upstream mic audio is **16 kHz** because that's what
Whisper expects; downstream TTS is **24 kHz** because that's CSM's native rate through Mimi. Don't
resample either to make them match — you'd discard quality on the way out and waste bandwidth on the
way in. `to_16k()` exists for exactly this reason.

**Client-side VAD.** A fixed-duration recorder makes you wait out the timer after you've stopped
talking, and that dead air *is* the thing users perceive as sluggishness:

> *"The VAD silence threshold — how long the user must stop talking before you decide they are
> finished — costs 200 to 300 ms, and users perceive it directly as sluggishness."*

So `record_mic()` watches microphone RMS in the browser and stops on ~700 ms of silence instead of
running to the clock. It's the cheapest large latency win available in a notebook. The guide's
better options — speculative LLM start at 100 ms, or a semantic turn-detection model that predicts
whether an utterance is syntactically *complete* rather than timing silence — need a streaming
architecture this notebook doesn't have.

In [ ]:
import base64, io, re, subprocess, tempfile, os
import numpy as np, soundfile as sf, librosa
from IPython.display import Audio, Javascript, display
from pathlib import Path


def _finalize(wav):
    """Watermark on the way out. Defined in Step 7c; no-op until then."""
    fn = globals().get("apply_watermark")
    return fn(wav) if fn is not None else wav


def play(wav, sr=SAMPLE_RATE, autoplay=True):
    """Render an audio player for a float32 numpy waveform."""
    display(Audio(np.asarray(_finalize(wav), dtype=np.float32), rate=sr, autoplay=autoplay))


def save_wav(wav, path, sr=SAMPLE_RATE):
    sf.write(path, np.asarray(_finalize(wav), dtype=np.float32), sr)
    return path


def load_wav(path, sr=SAMPLE_RATE):
    """Load anything ffmpeg/libsndfile understands as mono float32 at `sr`."""
    wav, _ = librosa.load(path, sr=sr, mono=True)
    return wav.astype(np.float32)


def to_16k(wav, sr=SAMPLE_RATE):
    """Whisper wants 16 kHz; CSM/Mimi want 24 kHz. Keep both around."""
    return librosa.resample(np.asarray(wav, dtype=np.float32), orig_sr=sr, target_sr=16_000)


def secs(wav, sr=SAMPLE_RATE):
    return len(wav) / sr


# --------------------------------------------------------------- microphone
# Records until you stop talking rather than until a timer expires. Falls back to the
# hard cap if the RMS gate never trips (noisy room, mic gain too low).
_RECORD_JS = """
var recordVAD = (maxMs, silenceMs, minMs, threshold) => new Promise(async resolve => {
  const stream   = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks   = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.onstop = () => {
    const blob   = new Blob(chunks);
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  };

  // WebAudio tap purely for the level meter -- MediaRecorder still owns the capture
  const ctx      = new AudioContext();
  const analyser = ctx.createAnalyser();
  analyser.fftSize = 1024;
  ctx.createMediaStreamSource(stream).connect(analyser);
  const buf = new Float32Array(analyser.fftSize);

  recorder.start();
  const t0 = Date.now();
  let lastVoice = t0, spoke = false;

  const tick = setInterval(() => {
    analyser.getFloatTimeDomainData(buf);
    let s = 0;
    for (let i = 0; i < buf.length; i++) s += buf[i] * buf[i];
    const rms = Math.sqrt(s / buf.length);
    const now = Date.now();
    if (rms > threshold) { lastVoice = now; spoke = true; }

    const elapsed = now - t0;
    // stop on trailing silence, but only after they've actually said something
    const done = elapsed > maxMs ||
                 (spoke && elapsed > minMs && (now - lastVoice) > silenceMs);
    if (done) {
      clearInterval(tick);
      recorder.stop();
      ctx.close();
      stream.getTracks().forEach(t => t.stop());
    }
  }, 50);
});
"""

VAD_SILENCE_MS = 700    # trailing silence that ends a turn. Lower = snappier, more early cuts
VAD_THRESHOLD  = 0.015  # RMS gate. Raise in a noisy room, lower if it cuts you off
VAD_MIN_MS     = 600    # ignore silence before this, so a slow start isn't clipped


def record_mic(seconds=12, use_vad=True, silence_ms=None, threshold=None):
    """Record from the browser mic -> 24 kHz mono float32.

    `seconds` is a hard ceiling, not a target: with VAD on, recording ends ~`silence_ms`
    after you stop talking. Returns None on failure so callers can fall back to typing.
    """
    from google.colab import output
    silence_ms = VAD_SILENCE_MS if silence_ms is None else silence_ms
    threshold  = VAD_THRESHOLD if threshold is None else threshold

    print(f"🔴 Listening (up to {seconds}s) ... speak now" if use_vad
          else f"🔴 Recording {seconds}s ...")
    display(Javascript(_RECORD_JS))
    try:
        data_url = output.eval_js(
            f"recordVAD({int(seconds * 1000)}, "
            f"{int(silence_ms) if use_vad else 10 ** 9}, "
            f"{int(VAD_MIN_MS)}, {float(threshold)})"
        )
    except Exception as e:
        print("Microphone unavailable:", e)
        return None
    if not data_url or "," not in data_url:
        print("No audio captured.")
        return None
    print("⏹️  Got it, transcribing ...")

    raw = base64.b64decode(data_url.split(",")[1])
    with tempfile.TemporaryDirectory() as d:
        src, dst = Path(d) / "in.webm", Path(d) / "out.wav"
        src.write_bytes(raw)
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
             "-ar", str(SAMPLE_RATE), "-ac", "1", str(dst)],
            check=True,
        )
        return load_wav(dst)


def transcribe(wav, sr=SAMPLE_RATE):
    out = asr(to_16k(wav, sr), generate_kwargs={"language": "en", "task": "transcribe"})
    return out["text"].strip()


print("Audio helpers ready.")

## Step 6 — The conversational context engine

**This is the part that matters.** Everything else in this notebook is off-the-shelf.

CSM's input is an interleaved sequence of `[speaker]text` tokens and Mimi audio frames. A turn that
carries its **audio** teaches the model how this conversation *sounds*; a turn carrying only text
teaches it only what was said. So we keep the waveforms.

Two things this class has to get right:

**Frame budgeting.** The backbone has 2048 positions. Audio spends 12.5 of them per second, so 60
seconds of history is 750 positions before a single word of text. We count text tokens exactly (via
the processor's tokenizer) and audio frames exactly (`ceil(samples / 24000 × 12.5) + 1` for the EOS
frame), then evict oldest-first until we fit — reserving room for what we're about to generate.

**Anchor pinning.** The voice anchor is *never* evicted. Drop it and the assistant's timbre resets
to a random stranger mid-conversation.

**A window, not a warehouse.** The frame budget is a *correctness* limit — it stops you exceeding
2048 positions. It is not a target to fill. Every segment costs prefill time, so the quality window
is deliberately much smaller: pinned reference plus the last **two to four turns**. Beyond that,
latency grows and quality does not. `MAX_CONTEXT_TURNS` is that knob, and it's why an hour-long
conversation doesn't get progressively slower.

In [ ]:
import math
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class Turn:
    """One utterance. `audio` is 24 kHz mono float32 -- keeping it is the whole point."""
    speaker: int
    text: str
    audio: Optional[np.ndarray] = None

    def cost(self) -> int:
        """Positions this turn consumes in the backbone's 2048-token window."""
        # text: "[spk]text" plus BOS/EOS framing
        n_text = len(csm_processor.tokenizer.encode(f"[{self.speaker}]{self.text}")) + 2
        n_audio = 0
        if self.audio is not None and len(self.audio) > 0:
            n_audio = math.ceil(len(self.audio) / SAMPLE_RATE * FRAME_RATE) + 1  # +1 = EOS frame
        return n_text + n_audio

    def to_message(self, with_audio: bool = True) -> dict:
        content = [{"type": "text", "text": self.text}]
        if with_audio and self.audio is not None and len(self.audio) > 0:
            # the processor accepts a raw numpy array under the "path" key
            content.append({"type": "audio", "path": np.asarray(self.audio, dtype=np.float32)})
        return {"role": str(self.speaker), "content": content}


class Conversation:
    """Rolling text+audio history for CSM, with a pinned voice anchor.

    Sesame's core claim: prosody should be inferred from conversational context, not guessed
    from an isolated sentence. That only works if the context still has audio in it.
    """

    def __init__(self, anchors: Optional[List[Turn]] = None, max_seq: int = CSM_MAX_SEQ,
                 max_turns: int = MAX_CONTEXT_TURNS):
        self.anchors: List[Turn] = list(anchors or [])   # never evicted
        self.turns: List[Turn] = []                      # windowed, then evicted oldest-first
        self.max_seq = max_seq
        self.max_turns = max_turns

    def add(self, turn: Turn):
        self.turns.append(turn)

    def add_anchor(self, turn: Turn):
        self.anchors.append(turn)

    def build(self, next_text: str, next_speaker: int, reserve_frames: int = 300,
              pending: Optional[List[Turn]] = None):
        """Return the message list for `apply_chat_template`, trimmed to fit.

        Two independent limits, and both matter:
          * `max_turns`  -- quality/latency. Every segment costs prefill time, and past a
                            handful of turns latency grows while quality does not.
          * `max_seq`    -- correctness. The backbone physically cannot exceed 2048 positions.

        `pending` is the in-flight reply's own chunks: exempt from the turn window (they're
        one utterance, not history) but still subject to the hard sequence budget.
        `reserve_frames` is headroom for the audio we're about to generate.
        """
        head = Turn(next_speaker, next_text, None)
        budget = self.max_seq - reserve_frames - head.cost() - 16   # 16 = safety margin
        budget -= sum(t.cost() for t in self.anchors)
        if budget < 0:
            raise ValueError(
                "Voice anchors alone overflow the context window -- use shorter anchor clips."
            )

        # quality window first: pinned reference + the last few turns, per the guide
        window = self.turns[-self.max_turns:] + list(pending or [])

        kept, used = [], 0
        for turn in reversed(window):              # newest first, oldest evicted
            c = turn.cost()
            if used + c > budget:
                # degrade to text-only before dropping the turn entirely
                text_only = Turn(turn.speaker, turn.text, None)
                if used + text_only.cost() <= budget:
                    kept.append(text_only)
                    used += text_only.cost()
                continue
            kept.append(turn)
            used += c
        kept.reverse()

        msgs = [t.to_message() for t in self.anchors]
        msgs += [t.to_message() for t in kept]
        msgs.append({"role": str(next_speaker), "content": [{"type": "text", "text": next_text}]})
        return msgs

    def transcript(self, n: int = 6):
        for t in self.turns[-n:]:
            who = "assistant" if t.speaker == ASSISTANT else "user"
            dur = f" [{secs(t.audio):.1f}s audio]" if t.audio is not None else " [text only]"
            print(f"  {who:>9}: {t.text}{dur}")


print("Conversation engine ready.")

## Step 7a — Text normalization

> *"The model receives raw text and will read it literally. A short normalisation pass in front of
> the TTS removes an entire class of embarrassing defects."*

This is ranked #5 in why voice projects fail, and the reason it survives so long is that it's
**invisible to the developer who wrote the string and glaring to the user who hears it**. `$1,500`
read as "dollar one comma five zero zero" is not a subtle defect.

Six categories, all handled below:

| Input | Spoken |
|---|---|
| `$1,500.50` | one thousand five hundred dollars and fifty cents |
| `12km`, `5°C`, `~40%` | twelve kilometres, five degrees Celsius, about forty percent |
| `NASA` vs `FBI` | "NASA" (a word) vs "F B I" (three letters) |
| `03/12`, `2026`, `3rd` | March twelfth, twenty twenty-six, third |
| `bob@mail.com`, URLs | bob at mail dot com, "a link" |
| `**bold**`, `- bullet`, 😄 | stripped as a backstop |

Initialisms are the one case that **cannot** be solved with a rule — only a lookup table knows that
NASA is a word and FBI is three letters. `SPOKEN_AS_WORD` below is that table; extend it for your
domain.

In [ ]:
import re

# num2words is nicer, but its `docopt` dependency fails to build on some images.
# Number-to-speech is too load-bearing to lose to a packaging hiccup, so there's a
# self-contained fallback. Identical interface either way.
try:
    from num2words import num2words as _n2w
except Exception:
    _n2w = None
    print("num2words unavailable -- using the built-in speller.")

_ONES = ("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
         "fifteen sixteen seventeen eighteen nineteen").split()
_TENS = ("", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety")
_SCALES = ((10**12, "trillion"), (10**9, "billion"), (10**6, "million"),
           (1000, "thousand"), (100, "hundred"))
_ORD_IRREG = {"one": "first", "two": "second", "three": "third", "five": "fifth",
              "eight": "eighth", "nine": "ninth", "twelve": "twelfth"}


def _int_words(n: int) -> str:
    n = int(n)
    if n < 0:
        return "minus " + _int_words(-n)
    if n < 20:
        return _ONES[n]
    if n < 100:
        t, r = divmod(n, 10)
        return _TENS[t] + (f"-{_ONES[r]}" if r else "")
    for value, name in _SCALES:
        if n >= value:
            head, rest = divmod(n, value)
            out = f"{_int_words(head)} {name}"
            return out + (f" and {_int_words(rest)}" if 0 < rest < 100
                          else f" {_int_words(rest)}" if rest else "")
    return _ONES[n]


def _ordinal_words(n: int) -> str:
    w = _int_words(int(n))
    last = re.split(r"[\s-]", w)[-1]
    repl = (_ORD_IRREG.get(last)
            or (last[:-1] + "ieth" if last.endswith("y") else last + "th"))
    return w[:len(w) - len(last)] + repl


def _year_words(y: int) -> str:
    y = int(y)
    if 1100 <= y < 2000 or 2010 <= y < 2100:
        hi, lo = divmod(y, 100)
        if lo == 0:
            return f"{_int_words(hi)} hundred"
        return f"{_int_words(hi)} oh {_int_words(lo)}" if lo < 10 else f"{_int_words(hi)} {_int_words(lo)}"
    return _int_words(y)


def num2words(n, to="cardinal"):
    """num2words when available, built-in speller otherwise."""
    if _n2w is not None:
        try:
            return _n2w(n, to=to)
        except Exception:
            pass
    if to == "ordinal":
        return _ordinal_words(n)
    if to == "year":
        return _year_words(n)
    s = str(n)
    if "." in s:                                    # 3.5 -> "three point five"
        whole, frac = s.split(".", 1)
        digits = " ".join(_ONES[int(d)] for d in frac if d.isdigit())
        return f"{_int_words(whole or 0)} point {digits}"
    return _int_words(s)


# ---- the lookup table only a human can write --------------------------------
SPOKEN_AS_WORD = {           # said as a word -> leave alone
    "NASA", "NATO", "UNESCO", "UNICEF", "OPEC", "SCUBA", "LASER", "RADAR",
    "AIDS", "PIN", "GIF", "JPEG", "PNG", "SQL", "ASCII", "RAM", "ROM", "ZIP",
    "CAPTCHA", "WYSIWYG", "MODEM", "SIM", "WIFI", "COVID",
}
SPOKEN_SPECIAL = {           # neither a word nor letters
    "OK": "okay", "OKAY": "okay", "AKA": "also known as", "ETA": "E T A",
    "ASAP": "as soon as possible", "FAQ": "F A Q", "IDK": "I don't know",
}

# Ambiguous short units (m, g, l, in) are deliberately absent: "5 in the morning"
# must not become "five inches the morning".
_UNITS = {
    "km": "kilometres", "kph": "kilometres per hour", "mph": "miles per hour",
    "kg": "kilograms", "mg": "milligrams", "lb": "pounds", "lbs": "pounds",
    "oz": "ounces", "cm": "centimetres", "mm": "millimetres", "ft": "feet",
    "mi": "miles", "ml": "millilitres", "hr": "hours", "hrs": "hours",
    "min": "minutes", "mins": "minutes", "sec": "seconds", "secs": "seconds",
    "GB": "gigabytes", "MB": "megabytes", "KB": "kilobytes", "TB": "terabytes",
}
_CURRENCY = {"$": ("dollars", "cents"), "£": ("pounds", "pence"),
             "€": ("euros", "cents"), "¥": ("yen", None)}
_SCALE = {"k": "thousand", "m": "million", "b": "billion", "t": "trillion"}
_MONTHS = ["", "January", "February", "March", "April", "May", "June", "July",
           "August", "September", "October", "November", "December"]

DATE_ORDER = "MDY"     # "03/12" -> March twelfth. Set "DMY" for the rest of the world.
SPEAK_URLS = False     # False -> "a link". True -> spelled out, which is usually worse.

_EMOJI = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF" "\U0001F1E6-\U0001F1FF" "←-⇿"
    "⬀-⯿" "️" "]+", flags=re.UNICODE
)


def _cardinal(n):
    try:
        return num2words(int(n)) if float(n) == int(float(n)) else num2words(float(n))
    except Exception:
        return str(n)


def _money(sym, amount, scale=None):
    major, minor = _CURRENCY.get(sym, ("dollars", "cents"))
    amount = amount.replace(",", "")
    if scale:
        return f"{_cardinal(amount)} {_SCALE[scale.lower()]} {major}"
    if "." in amount:
        whole, frac = amount.split(".", 1)
        frac = (frac + "00")[:2]
        if int(frac) and minor:
            return f"{_cardinal(whole)} {major} and {_cardinal(frac)} {minor}"
        return f"{_cardinal(whole)} {major}"
    return f"{_cardinal(amount)} {major}"


def _spell(acronym):
    if acronym in SPOKEN_SPECIAL:
        return SPOKEN_SPECIAL[acronym]
    if acronym in SPOKEN_AS_WORD:
        return acronym
    return " ".join(acronym)


def normalize_for_speech(text: str) -> str:
    """Expand everything a TTS would otherwise read literally. Order matters."""
    t = text

    # 1. URLs / emails / paths -- decide once, never let the model improvise
    t = re.sub(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b",
               lambda m: m.group(0).replace("@", " at ").replace(".", " dot "), t)
    if SPEAK_URLS:
        t = re.sub(r"https?://\S+|\bwww\.\S+",
                   lambda m: m.group(0).replace(".", " dot ").replace("/", " slash "), t)
    else:
        t = re.sub(r"https?://\S+|\bwww\.\S+", " a link ", t)
    # first segment must start with a letter, or "12/25/2026" reads as a file path
    t = re.sub(r"(?<![\w/])/[A-Za-z][\w.-]*(?:/[\w.-]+)+/?", " a file path ", t)

    # 2. currency (before plain numbers)
    t = re.sub(r"([$£€¥])\s?([\d,]+(?:\.\d{1,2})?)\s*([kKmMbBtT])?\b",
               lambda m: _money(m.group(1), m.group(2), m.group(3)), t)

    # 3. percent, temperature
    t = re.sub(r"([\d,]+(?:\.\d+)?)\s?%", lambda m: f"{_cardinal(m.group(1).replace(',', ''))} percent", t)
    t = re.sub(r"(-?[\d.]+)\s?°\s?([CF])",
               lambda m: f"{_cardinal(m.group(1))} degrees "
                         f"{'Celsius' if m.group(2) == 'C' else 'Fahrenheit'}", t)
    t = re.sub(r"(-?[\d.]+)\s?°", lambda m: f"{_cardinal(m.group(1))} degrees", t)

    # 4. clock times
    def _time(m):
        h, mi, ap = int(m.group(1)), m.group(2), (m.group(3) or "").lower().replace(".", "")
        said = _cardinal(h) + (" o'clock" if mi == "00" else
                               f" oh {_cardinal(mi[1])}" if mi[0] == "0" else f" {_cardinal(mi)}")
        return said + (f" {ap[0]} {ap[1]}" if ap else "")
    t = re.sub(r"\b(\d{1,2}):(\d{2})\s?([ap]\.?m\.?)?\b", _time, t, flags=re.I)

    # 5. numeric dates -- ambiguous, so resolve by convention rather than guessing
    def _date(m):
        a, b = int(m.group(1)), int(m.group(2))
        mth, day = (a, b) if DATE_ORDER == "MDY" else (b, a)
        if not (1 <= mth <= 12 and 1 <= day <= 31):
            return m.group(0)
        out = f"{_MONTHS[mth]} {num2words(day, to='ordinal')}"
        if m.group(3):
            y = int(m.group(3)); y += 2000 if y < 100 else 0
            out += f" {num2words(y, to='year')}"
        return out
    t = re.sub(r"\b(\d{1,2})/(\d{1,2})(?:/(\d{2,4}))?\b", _date, t)

    # 6. ordinals, years, ranges
    t = re.sub(r"\b(\d+)(st|nd|rd|th)\b",
               lambda m: num2words(int(m.group(1)), to="ordinal"), t, flags=re.I)
    t = re.sub(r"\b(1[89]\d{2}|20\d{2})\b", lambda m: num2words(int(m.group(1)), to="year"), t)

    # 7. ranges -- units first, or "10-20 km" loses the "km" when the range eats the digits
    _u = "|".join(sorted(_UNITS, key=len, reverse=True))
    t = re.sub(r"\b(\d[\d,]*)\s?[-–]\s?(\d[\d,]*)\s?(" + _u + r")\b",
               lambda m: f"{_cardinal(m.group(1).replace(',', ''))} to "
                         f"{_cardinal(m.group(2).replace(',', ''))} {_UNITS[m.group(3)]}", t)
    t = re.sub(r"\b(\d[\d,]*)\s?[-–]\s?(\d[\d,]*)\b",
               lambda m: f"{_cardinal(m.group(1).replace(',', ''))} to "
                         f"{_cardinal(m.group(2).replace(',', ''))}", t)

    # 8. units, then bare numbers
    unit_re = r"\b(\d[\d,]*(?:\.\d+)?)\s?(" + _u + r")\b"
    t = re.sub(unit_re, lambda m: f"{_cardinal(m.group(1).replace(',', ''))} {_UNITS[m.group(2)]}", t)
    t = re.sub(r"\b\d[\d,]*(?:\.\d+)?\b", lambda m: _cardinal(m.group(0).replace(",", "")), t)

    # 9. initialisms -- the lookup table decides
    t = re.sub(r"\b[A-Z]{2,6}\b", lambda m: _spell(m.group(0)), t)

    # 10. loose symbols
    for sym, word in ((" & ", " and "), (" + ", " plus "), (" = ", " equals "),
                      ("~", "about "), (" @ ", " at ")):
        t = t.replace(sym, word)
    return t


def clean_for_speech(text: str, normalize: bool = True) -> str:
    """Markdown/emoji backstop, then full normalization."""
    t = _EMOJI.sub("", text)
    t = re.sub(r"```.*?```", " ", t, flags=re.DOTALL)          # code fences
    t = re.sub(r"^\s*[-•+*]\s+", "", t, flags=re.MULTILINE)    # bullets (before markdown strip)
    t = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", t)                  # links -> label
    t = re.sub(r"[*_`#>|]+", " ", t)                           # markdown syntax
    t = re.sub(r"\s*\([^)]{0,80}\)", "", t)                    # parenthetical asides have no spoken form
    if normalize:
        t = normalize_for_speech(t)
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"\s+([,.!?;:])", r"\1", t)
    return t.strip()


for demo in ["That'll be $1,500.50, due 03/12 before 5:30pm.",
             "The FBI and NASA confirmed a 40% rise to 12km by 2026.",
             "**Email** bob@mail.com — see https://x.com/docs (details there)."]:
    print(f"  {demo}\n    -> {clean_for_speech(demo)}\n")

## Step 7b — The chunker

> *"Between the LLM and the TTS sits a small state machine that decides where to cut. Get it wrong
> and every other thing in this document stops mattering."*

Splitting on `(?<=[.!?…])\s+` — the obvious approach, and what this notebook did before — has two
bugs that only show up in real text:

- **`It costs 3.5 million`** splits after `3.` and CSM synthesizes two utterances with a full stop
  between them. You get "three point" ... "five million".
- **`Dr. Smith called`** splits after `Dr.`

The guide's state machine cuts **where a person would breathe**, at the earliest legal boundary:

| Flush when | Threshold |
|---|---|
| Primary delimiter `. ? ! …` | ~20 chars in buffer |
| Delimiter **followed by** a discourse marker | minimum length only |
| Plain secondary `, ; :` | past ~60 chars, so you never emit two-word fragments |
| Hard ceiling | ~180 chars, cut at the last word boundary |

**Never** flush when the period sits between digits, ends a known abbreviation, or is the last
character in the buffer.

That last rule is subtle and worth understanding, because it's the one that never shows up in
offline tests: at the tail of a *stream* you cannot yet distinguish `3.` with more digits coming
from `3.` ending a sentence, nor a full stop from the first dot of an ellipsis. Deferring by one
token costs ~30 ms and prevents the bug. This chunker is written incrementally — `push()` / `flush()`
— so it behaves identically offline and streaming, and ports straight to a WebSocket deployment.

**The correction that matters most.** The guide flags that treating bare discourse markers as
standalone flush triggers produces *worse* output than no chunking at all: `look`, `right`, `well`
and `so` are ordinary content words, and the naive rule cuts "Let me take a look at that" after
"take a." A marker is **evidence about a delimiter that already exists** — never a boundary of its
own. That's why `_marker_follows` is only ever consulted at a character that is already punctuation.

In [ ]:
class SpeechChunker:
    """Incremental text -> speakable chunks. Cut where a person would breathe.

    Offline:   list(chunker.push(whole_text)) + list(chunker.flush())
    Streaming: call push() per LLM token, flush() at end of turn.
    """

    PRIMARY   = ".?!…"
    SECONDARY = ",;:"
    MARKERS   = ("so", "anyway", "but", "and", "well", "right", "look", "actually",
                 "basically", "then", "because", "although", "though", "okay")
    ABBREV    = {"mr", "mrs", "ms", "dr", "prof", "sr", "jr", "st", "mt", "vs", "etc",
                 "approx", "dept", "est", "fig", "no", "vol", "inc", "ltd", "co",
                 "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep", "sept",
                 "oct", "nov", "dec", "e.g", "i.e", "a.m", "p.m", "u.s"}

    def __init__(self, min_chars: int = 20, secondary_min: int = 60, hard_max: int = 180):
        self.min_chars, self.secondary_min, self.hard_max = min_chars, secondary_min, hard_max
        self._buf = ""
        self.emitted = 0

    def push(self, text: str):
        self._buf += text
        yield from self._drain(final=False)

    def flush(self):
        yield from self._drain(final=True)
        tail = self._buf.strip()
        self._buf = ""
        if tail:                      # the final chunk may be shorter than min_chars
            self.emitted += 1
            yield tail

    # ------------------------------------------------------------------ internals
    def _drain(self, final: bool):
        while True:
            i = self._boundary(final)
            if i is None:
                return
            chunk, self._buf = self._buf[:i + 1].strip(), self._buf[i + 1:].lstrip()
            if chunk:
                self.emitted += 1
                yield chunk

    def _boundary(self, final: bool):
        buf, n = self._buf, len(self._buf)
        for i, ch in enumerate(buf):
            if ch not in self.PRIMARY and ch not in self.SECONDARY:
                continue
            if not self._legal(buf, i, n, final):
                continue
            length = len(buf[:i + 1].strip())
            if ch in self.PRIMARY:
                if length >= self.min_chars:
                    return i                      # earliest legal boundary wins
            else:
                need = self.min_chars if self._marker_follows(buf, i) else self.secondary_min
                if length >= need:
                    return i
        if n >= self.hard_max:                    # runaway clause: cut at a word boundary
            cut = buf.rfind(" ", 0, self.hard_max)
            if cut > self.min_chars:
                return cut
        return None

    def _legal(self, buf, i, n, final) -> bool:
        # streaming: never decide on the last char -- "3." might become "3.5"
        if not final and i == n - 1:
            return False
        if buf[i] == ".":
            if 0 < i < n - 1 and buf[i - 1].isdigit() and buf[i + 1].isdigit():
                return False                      # 3.5
            if i < n - 1 and buf[i + 1] == ".":
                return False                      # mid-ellipsis
            word = re.split(r"[\s(\"']", buf[:i])[-1].lower()
            if word in self.ABBREV:
                return False                      # Dr. / etc.
            if len(word) == 1 and word.isalpha():
                return False                      # initial, as in "J. Smith"
        return True

    def _marker_follows(self, buf, i) -> bool:
        """Markers lower the bar on punctuation that is ALREADY there. Never a boundary alone."""
        rest = buf[i + 1:].lstrip().lower()
        return any(rest.startswith(m) and (len(rest) == len(m) or not rest[len(m)].isalpha())
                   for m in self.MARKERS)


def split_for_speech(text: str, **kw) -> List[str]:
    c = SpeechChunker(**kw)
    return list(c.push(text)) + list(c.flush())


for demo in ["It costs 3.5 million dollars, which is well over budget.",
             "Dr. Smith called. She wants the report by Friday, so I said yes.",
             "Let me take a look at that and see what I can find for you today.",
             "Sure."]:
    print(f"  {demo}\n    -> {split_for_speech(demo)}\n")

## Step 7c — Synthesis

Three things beyond the plain generate call:

**Chunk-boundary fades.** Concatenating two independently generated waveforms leaves a
discontinuity at the join, and a discontinuity is a broadband transient — users hear a click and
read it as a defect even when they can't name what went wrong. A short linear ramp is inaudible as
a fade and removes the artefact completely. The guide prescribes 15 ms for interruption cuts; 10 ms
at joins does the same job here.

**Bounded generation.** A runaway generation can stall a whole turn, so `max_new_tokens` is capped
from the estimated duration rather than left open.

**Watermarking, kept on.** CSM ships with a watermarker and there is community advice about removing
it for a performance gain. Don't:

> *"It is the mechanism that makes synthetic speech identifiable, it is an explicit condition of the
> model's terms of use, and AI-transparency rules are moving toward requiring exactly this. Buy the
> performance back with `torch.compile` instead."*

The 🤗 Transformers integration doesn't apply it, so this notebook adds it back explicitly. Context
audio stays clean and only what you hear or save is watermarked, so the mark never compounds through
the conversation history. **The bundled key is public and therefore not secure** — generate your own
and keep it secret if you deploy this.

In [ ]:
WATERMARK = True
# Public key from Sesame's repo: identifies audio as CSM-generated, secure against nothing.
# Deploying? Generate your own five-integer key and keep it private.
WATERMARK_KEY = [212, 211, 146, 56, 201]

_watermarker = None
if WATERMARK:
    try:
        import silentcipher
        _watermarker = silentcipher.get_model(model_type="44.1k", device=DEVICE)
        print("Watermarker loaded.")
    except Exception as e:
        print(f"!! silentcipher unavailable ({type(e).__name__}) -- output will NOT be watermarked.")
        print("   Fine for local experiments; fix it before you ship anything.")


@torch.inference_mode()
def apply_watermark(wav: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    if _watermarker is None or wav is None or len(wav) == 0:
        return wav
    try:
        import torchaudio
        t = torch.as_tensor(np.asarray(wav, dtype=np.float32), device=DEVICE)
        t44 = torchaudio.functional.resample(t, orig_freq=sr, new_freq=44100)
        enc, _ = _watermarker.encode_wav(t44, 44100, WATERMARK_KEY, calc_sdr=False, message_sdr=36)
        out = torchaudio.functional.resample(enc, orig_freq=44100, new_freq=min(44100, sr))
        return out.detach().float().cpu().numpy()
    except Exception as e:
        print(f"   (watermark skipped: {type(e).__name__})")
        return wav


def fade_edges(wav: np.ndarray, ms: float = 10.0, sr: int = SAMPLE_RATE) -> np.ndarray:
    """Linear ramp in/out. Kills the click at a concatenation seam."""
    n = int(sr * ms / 1000)
    if wav is None or len(wav) < 2 * n or n < 2:
        return wav
    w = np.array(wav, dtype=np.float32, copy=True)
    w[:n]  *= np.linspace(0.0, 1.0, n, dtype=np.float32)
    w[-n:] *= np.linspace(1.0, 0.0, n, dtype=np.float32)
    return w


@torch.inference_mode()
def csm_generate(messages, max_seconds: float = 15.0, temperature: float = 0.9,
                 top_k: int = 50) -> np.ndarray:
    """One CSM forward pass: messages -> waveform.

    Backbone samples codebook 0 (semantic); the depth decoder samples codebooks 1-31
    (acoustic). Both get their own sampling params -- `depth_decoder_*` kwargs.
    """
    inputs = csm_processor.apply_chat_template(messages, tokenize=True, return_dict=True)
    inputs = inputs.to(DEVICE)
    if inputs.get("input_values") is not None:
        inputs["input_values"] = inputs["input_values"].to(CSM_DTYPE)

    audio = csm_model.generate(
        **inputs,
        output_audio=True,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        max_new_tokens=int(max_seconds * FRAME_RATE),
        depth_decoder_do_sample=True,
        depth_decoder_temperature=temperature,
        depth_decoder_top_k=top_k,
    )
    return audio[0].float().cpu().numpy()


def speak(text: str, conv: Conversation, speaker: int = ASSISTANT,
          temperature: float = 0.9, verbose: bool = True, normalize: bool = True):
    """Synthesize `text` in conversational context, chunk by chunk.

    Returns (full_waveform, [Turn, ...]). The turns are context-shaped and un-watermarked,
    so the caller appends them and the next reply hears this one cleanly.
    """
    chunks = split_for_speech(clean_for_speech(text, normalize=normalize))
    pieces, new_turns = [], []
    t_start = time.time()

    for i, chunk in enumerate(chunks):
        est = min(15.0, max(3.0, len(chunk) / 13.0))       # ~13 chars/sec of speech
        # `pending` = earlier chunks of THIS reply: always kept, so prosody carries
        # across the seam even though the turn window is small.
        msgs = conv.build(chunk, speaker,
                          reserve_frames=int(est * FRAME_RATE) + 40,
                          pending=new_turns)

        wav = csm_generate(msgs, max_seconds=est + 3.0, temperature=temperature)
        new_turns.append(Turn(speaker, chunk, wav))         # clean copy for context
        pieces.append(fade_edges(wav))                      # faded copy for playback

        if verbose and i == 0:
            print(f"   ⏱️  first audio in {time.time()-t_start:.1f}s")

    full = np.concatenate(pieces) if pieces else np.zeros(0, dtype=np.float32)
    if verbose and len(full):
        gen = time.time() - t_start
        rtf = gen / max(secs(full), 1e-6)
        flag = "" if rtf < 1.0 else "   ⚠️  slower than real time"
        print(f"   ⏱️  {secs(full):.1f}s of audio in {gen:.1f}s "
              f"(RTF {rtf:.2f}x, {len(chunks)} chunk(s)){flag}")
    return full, new_turns


print("Synthesis ready.")

## Step 8 — Bootstrap a voice anchor

CSM-1B ships as a **base** model. It was never fine-tuned on a specific voice, and with an empty
context it samples a fresh speaker identity every single call — your assistant would sound like a
different person in every sentence. (Sesame's actual demo runs a *fine-tuned* variant. That's the
one real difference between this notebook and their product.)

The fix without any training: generate one utterance, decide you like it, and pin it as context
forever. Every later generation is conditioned on it and inherits the timbre.

Run the cell, listen, and **re-run with a different `VOICE_SEED` until you find a voice you like.**
This is the single highest-leverage knob in the notebook.

In [ ]:
VOICE_SEED = 42                # <-- change this and re-run until you like the voice
ANCHOR_TEXT = ("Hey, good to hear from you. I'm doing great, thanks for asking. "
               "So what did you want to talk about today?")

torch.manual_seed(VOICE_SEED)
np.random.seed(VOICE_SEED)

print(f"Bootstrapping voice with seed {VOICE_SEED} ...")
_boot = Conversation()
anchor_wav = csm_generate(
    [{"role": str(ASSISTANT), "content": [{"type": "text", "text": ANCHOR_TEXT}]}],
    max_seconds=12.0, temperature=0.85,
)

VOICE_ANCHOR = Turn(ASSISTANT, ANCHOR_TEXT, anchor_wav)
print(f"Anchor: {secs(anchor_wav):.1f}s, {VOICE_ANCHOR.cost()} context positions")
save_wav(anchor_wav, "voice_anchor.wav")
play(anchor_wav)

### Optional — use *your own* voice as the anchor

Record ~10 seconds instead of bootstrapping. This is voice cloning, so: **only record yourself, or
someone who has actually agreed to it.** Sesame's license prohibits the rest, and so does common
decency.

Skip this cell to keep the bootstrapped voice.

In [ ]:
USE_MY_VOICE = False   # flip to True, then run

if USE_MY_VOICE:
    my_wav = record_mic(seconds=10)
    if my_wav is not None:
        my_text = transcribe(my_wav)
        print("Transcript:", my_text)
        VOICE_ANCHOR = Turn(ASSISTANT, my_text, my_wav)
        print(f"Anchor replaced: {secs(my_wav):.1f}s, {VOICE_ANCHOR.cost()} positions")
        play(my_wav, autoplay=False)
else:
    print("Keeping the bootstrapped anchor. Set USE_MY_VOICE = True to override.")

## Step 9 — Hear the thesis: context vs. no context

Sesame's central claim in one experiment. Same sentence, twice:

- **A** — no context. The model has nothing to go on and gives you the safe, flat, average reading.
- **B** — with an excited, fast-paced conversation in the context window. Same words, different
  delivery, because the model can now infer *how* this conversation is going.

If B doesn't sound noticeably more alive than A, your anchor is weak — go back and reroll the seed.

In [ ]:
TEST_LINE = "Wait, are you serious? That is completely unbelievable."

print("── A: no context ─────────────────────────────────────────")
wav_a = csm_generate(
    [{"role": str(ASSISTANT), "content": [{"type": "text", "text": TEST_LINE}]}],
    max_seconds=8.0,
)
play(wav_a, autoplay=False)

print("\n── B: same line, inside an excited conversation ──────────")
ctx = Conversation(anchors=[VOICE_ANCHOR])

lead_in = [
    (USER,      "Okay I have to tell you something, you are not going to believe this."),
    (ASSISTANT, "Oh no. What happened?"),
    (USER,      "I just found out we got the grant. The whole thing. Every penny we asked for!"),
]
for spk, line in lead_in:
    w, turns = speak(line, ctx, speaker=spk, verbose=False)
    ctx.turns.extend(turns)

msgs = ctx.build(TEST_LINE, ASSISTANT, reserve_frames=120)
wav_b = csm_generate(msgs, max_seconds=8.0)
play(wav_b, autoplay=False)

print("\nSame text, same weights. The only difference is what was in the context window.")

## Step 10 — Gemma as the conversational brain

CSM generates speech; it cannot generate text (Sesame is explicit about this in their FAQ — it's an
audio model, not a general multimodal LLM). So Gemma writes the words.

> *"This section is worth more to perceived quality than anything else in the document, and it is
> usually the last thing teams get to."*

Here's the reframe that makes it click: **you are not prompting the LLM to write text. You are
prompting it to write a script that a person will speak aloud.** Those are different tasks, and
every chat model's default output is tuned for the wrong one — bullets, headers, parentheticals,
long subordinate clauses, hedges. Fed to any TTS that produces something like an encyclopedia entry
read under duress. The model isn't the problem. The text is.

The prompt below is the guide's reference prompt. Two rules in it do more work than the rest:

- **"Answer first, elaborate second."** If the user interrupts after two seconds they should already
  have the answer. Design for interruption.
- **"Put a comma where a speaker would breathe."** Punctuation is prosody, not grammar — it's your
  only direct handle on pacing:

| Mark | Effect | Use for |
|---|---|---|
| `.` | Full stop, pitch drop, ~400 ms pause | End of a thought |
| `,` | Short pause, pitch sustained | Clause break, list items |
| `...` | Long hesitant pause, trailing pitch | Thinking, softening, uncertainty |
| `?` | Rising terminal pitch | Real questions only — overuse sounds anxious |
| `!` | Raised energy | Sparingly; three in a row sounds manic |
| `—` | Abrupt break | Self-correction |

⚠️ **Do not evaluate a prompt change by reading the output.** Text that reads well and text that
sounds well are different, and you cannot tell them apart on a screen. Step 13 is your two-line
harness — play every change.

`llm_reply()` has the same signature on both backends, so everything downstream is backend-agnostic.
Three things the API path has to handle that the local path doesn't:

1. **No system instructions.** Gemma on the Gemini API 400s on `system_instruction`. We fold the
   prompt into the *most recent* user turn rather than the first — style adherence decays as a
   one-shot instruction scrolls out of view, and voice turns are short enough that re-sending it
   costs nothing.
2. **Different role names.** The Gemini API says `model`; Transformers says `assistant`.
3. **Rate limits.** Free tier is ~15 req/min, and a brisk conversation *will* hit that. 429s get
   retried with exponential backoff instead of crashing your chat.

In [ ]:
SYSTEM_PROMPT = """You are a voice assistant. Everything you say is spoken aloud, never read.
Write the way people talk, not the way people write.

- One idea per sentence. Under fifteen words.
- Two or three sentences total, unless more is requested.
- Answer first. Elaborate only after.
- No markdown, lists, headers, formatting, or emoji.
- No parentheses, no asides.
- Spell numbers as words.
- Put a comma where a speaker would breathe.
- An occasional "um" or "well" is fine. Not every sentence.
- If you didn't understand, ask one short question. Don't guess.

Tone: a friend, not a support agent. Direct, warm, no unnecessary apologies."""


@torch.inference_mode()
def _llm_reply_local(history, max_new_tokens=110, temperature=0.8) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history

    inputs = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(llm_model.device)

    out = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=llm_tokenizer.eos_token_id,
    )
    return llm_tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def _llm_reply_api(history, max_new_tokens=110, temperature=0.8, max_retries=4) -> str:
    from google.genai import types

    # Gemini API roles are "user" / "model" (not "assistant")
    contents = [
        types.Content(role="user" if m["role"] == "user" else "model",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]

    # Gemma rejects system_instruction -> fold it into the latest user turn.
    for c in reversed(contents):
        if c.role == "user":
            c.parts[0].text = f"{SYSTEM_PROMPT}\n\n---\n\nThey said: {c.parts[0].text}"
            break

    cfg = types.GenerateContentConfig(
        temperature=temperature,
        top_p=0.95,
        max_output_tokens=max_new_tokens,
    )

    for attempt in range(max_retries):
        try:
            r = genai_client.models.generate_content(
                model=GEMMA_API_MODEL, contents=contents, config=cfg)
            return r.text or ""
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg or "RESOURCE_EXHAUSTED" in msg.upper()
                         or "503" in msg or "UNAVAILABLE" in msg.upper())
            if transient and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"   ⏳ rate limited / unavailable, retrying in {wait}s "
                      f"({attempt + 1}/{max_retries})")
                time.sleep(wait)
                continue
            if "Developer instruction" in msg:
                raise RuntimeError(
                    "Gemma rejected a system instruction. Something is passing system_instruction "
                    "-- fold the prompt into a user turn instead."
                ) from e
            raise
    return ""


def llm_reply(history: List[dict], max_new_tokens: int = 110, temperature: float = 0.8) -> str:
    """history: [{"role": "user"/"assistant", "content": str}, ...] -> speakable text.

    Identical contract on both backends, so nothing downstream has to care.
    """
    fn = _llm_reply_api if LLM_BACKEND == "api" else _llm_reply_local
    reply = fn(history, max_new_tokens=max_new_tokens, temperature=temperature)
    reply = clean_for_speech(reply)
    if not reply:
        # empty happens on safety blocks or a truncated response -- don't hand "" to CSM
        reply = "Sorry, I lost my train of thought there. What were we saying?"
    return reply


t0 = time.time()
print("Gemma:", llm_reply([{"role": "user", "content": "Hey, what have you been up to?"}]))
print(f"   ⏱️  {time.time()-t0:.2f}s via {LLM_BACKEND}")

## Step 11 — Wire it together

The full loop. Note what gets stored where:

- **Gemma's history** is text only — that's all an LLM can use.
- **CSM's context keeps the waveforms**, including *your recorded voice* for user turns. The model
  hears the actual conversation, not a transcript of it. That's rule #1 from the paper, and it's why
  the speaker-id tokens matter: `[1]` marks your turns so CSM knows whose voice is whose and doesn't
  start answering in your voice.

In [ ]:
class VoiceAgent:
    """Whisper -> Gemma 3 -> CSM, sharing one conversation that remembers audio."""

    def __init__(self, anchor: Turn, temperature_llm=0.8, temperature_tts=0.9):
        self.ctx = Conversation(anchors=[anchor])   # CSM: text + audio
        self.history: List[dict] = []               # Gemma: text only
        self.t_llm = temperature_llm
        self.t_tts = temperature_tts

    def turn(self, user_text: str, user_audio: Optional[np.ndarray] = None,
             autoplay: bool = True):
        # 1. remember what the user said -- with their real audio if we have it
        self.ctx.add(Turn(USER, user_text, user_audio))
        self.history.append({"role": "user", "content": user_text})

        # 2. think
        t0 = time.time()
        reply = llm_reply(self.history, temperature=self.t_llm)
        print(f"🤖 {reply}")
        print(f"   ⏱️  Gemma: {time.time()-t0:.1f}s")

        # 3. speak, conditioned on everything above
        wav, new_turns = speak(reply, self.ctx, ASSISTANT, temperature=self.t_tts)

        # 4. remember our own voice too -- next reply hears this one
        self.ctx.turns.extend(new_turns)
        self.history.append({"role": "assistant", "content": reply})

        if len(wav):
            play(wav, autoplay=autoplay)
        return reply, wav

    def say_text(self, user_text: str, **kw):
        """Typed input -- no user audio in context."""
        return self.turn(user_text, None, **kw)

    def say_voice(self, seconds: int = 12, **kw):
        """Spoken input -- the user's waveform goes into CSM's context.

        `seconds` is the hard ceiling; VAD ends the turn when you stop talking.
        """
        wav = record_mic(seconds)
        if wav is None:
            print("No audio captured.")
            return None, None
        text = transcribe(wav)
        if not text:
            print("Didn't catch that.")
            return None, None
        print(f"🗣️  You: {text}")
        return self.turn(text, wav, **kw)

    def reset(self):
        self.ctx.turns.clear()
        self.history.clear()
        print("Conversation cleared (voice anchor kept).")


agent = VoiceAgent(VOICE_ANCHOR)
print("Agent ready.")

## Step 12 — Talk to it 🎤

Run the cell, allow microphone access, and speak when it says *Listening*. **Just stop talking when
you're done** — VAD ends the turn, you don't wait out a timer. Re-run for each turn; the context and
the voice persist across cells.

If it cuts you off mid-sentence, raise `VAD_SILENCE_MS`. If it never stops, your mic is noisier than
the gate — raise `VAD_THRESHOLD`.

**If the mic doesn't work** in your browser, skip to Step 13 and type instead.

In [ ]:
MAX_RECORD_SECONDS = 12   # ceiling only -- VAD normally ends the turn well before this

agent.say_voice(seconds=MAX_RECORD_SECONDS)

## Step 13 — Or just type

Same pipeline minus Whisper. Useful for iterating fast, and for checking the assistant's voice stays
put across many turns.

In [ ]:
agent.say_text("Hi! Can you tell me what makes your voice sound different from normal text to speech?")

In [ ]:
# keep going -- watch the voice stay consistent and the prosody react to the conversation
agent.say_text("That's interesting. Does it get better the longer we talk?")

In [ ]:
print("Context window usage:")
used = sum(t.cost() for t in agent.ctx.anchors) + sum(t.cost() for t in agent.ctx.turns)
print(f"  {used} / {CSM_MAX_SEQ} positions "
      f"({sum(secs(t.audio) for t in agent.ctx.turns if t.audio is not None):.1f}s of audio history)")
print("\nRecent turns:")
agent.ctx.transcript(n=8)

# agent.reset()   # clear history, keep the voice

## Step 14 — Two-speaker dialogue

CSM's original demo generates a *conversation*, not a monologue: two anchors, two speaker ids, one
shared context. Each line is conditioned on every line before it, from both speakers — which is why
the turns actually respond to each other's energy instead of sounding like two separate TTS runs
stitched together.

In [ ]:
DIALOGUE_SEED_B = 1234   # reroll for a different second voice

torch.manual_seed(DIALOGUE_SEED_B)
anchor_b_text = "Honestly? I wasn't expecting that at all. Tell me more about it."
anchor_b_wav = csm_generate(
    [{"role": str(USER), "content": [{"type": "text", "text": anchor_b_text}]}],
    max_seconds=10.0, temperature=0.85,
)
ANCHOR_B = Turn(USER, anchor_b_text, anchor_b_wav)
print("Second voice:"); play(anchor_b_wav, autoplay=False)

script = [
    (ASSISTANT, "So I finally tried that place you kept talking about."),
    (USER,      "No way. And? Was I right or was I right?"),
    (ASSISTANT, "You were right. I hate that you were right, but you were right."),
    (USER,      "I'm framing this moment. Say it one more time for me."),
    (ASSISTANT, "Not a chance. Let's just say the pasta was worth the wait."),
]

dialogue = Conversation(anchors=[VOICE_ANCHOR, ANCHOR_B])
pieces = []
for spk, line in script:
    who = "A" if spk == ASSISTANT else "B"
    print(f"{who}: {line}")
    wav, turns = speak(line, dialogue, speaker=spk, verbose=False)
    dialogue.turns.extend(turns)
    pieces.append(wav)
    pieces.append(np.zeros(int(0.25 * SAMPLE_RATE), dtype=np.float32))  # beat between turns

full = np.concatenate(pieces)
save_wav(full, "dialogue.wav")
print(f"\n{secs(full):.1f}s of dialogue -> dialogue.wav")
play(full, autoplay=False)

## Step 15 — (Optional) Fine-tuning, and the compute-amortization trick

Anchoring gets you a *consistent* voice. Fine-tuning gets you a *specific* one — that's the gap
between this notebook and Sesame's product, which runs a fine-tuned CSM variant.

This cell demonstrates the paper's training scheme, including the memory trick that makes it
tractable:

> The audio decoder processes an effective batch of `B × S × N` (batch × sequence × 32 codebooks),
> which is enormous. Sesame train it on a random **1/16 subset of frames** while codebook 0 trains
> on **every** frame — and report no perceivable difference in decoder loss.

In 🤗 Transformers that's exactly one argument: **`depth_decoder_labels_ratio=1/16`**. The processor
marks the skipped frames with `-101`, so the backbone still sees them and only the depth decoder
skips.

⚠️ **This is a mechanics demo, not a real training run.** It uses LoRA and a handful of steps on a
tiny dataset so it fits a free T4. A voice you'd actually ship needs hours of clean single-speaker
audio. **Restart the runtime before going back to inference** — the LoRA-wrapped model is not the
one the cells above expect.

In [ ]:
RUN_FINETUNE_DEMO = False   # flip to True to run

if RUN_FINETUNE_DEMO:
    from peft import LoraConfig, get_peft_model
    from datasets import load_dataset, Audio

    # swap in your own dataset: needs `text`, `audio`, `speaker_id` columns at 24 kHz
    ds = load_dataset("hf-internal-testing/dailytalk-dummy", split="train")
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    conversation = [
        {"role": f"{spk}",
         "content": [{"type": "text", "text": txt},
                     {"type": "audio", "path": aud["array"]}]}
        for txt, aud, spk in zip(ds[:4]["text"], ds[:4]["audio"], ds[:4]["speaker_id"])
    ]

    csm_model.train()
    csm_model.codec_model.eval()          # never train the codec
    for p in csm_model.codec_model.parameters():
        p.requires_grad_(False)

    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    peft_model = get_peft_model(csm_model, lora)
    peft_model.print_trainable_parameters()

    # ---- the compute-amortization knob from the paper -----------------------
    AMORTIZATION_RATIO = 1 / 16   # 1.0 = train the depth decoder on every frame (memory-hungry)

    inputs = csm_processor.apply_chat_template(
        conversation, tokenize=True, return_dict=True,
        output_labels=True,
        depth_decoder_labels_ratio=AMORTIZATION_RATIO,
    ).to(DEVICE)
    if inputs.get("input_values") is not None:
        inputs["input_values"] = inputs["input_values"].to(CSM_DTYPE)

    opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4)

    for step in range(5):
        out = peft_model(**inputs)
        out.loss.backward()
        opt.step(); opt.zero_grad()
        print(f"step {step}  total {out.loss.item():.4f}  "
              f"backbone {out.backbone_loss.item():.4f}  depth {out.depth_decoder_loss.item():.4f}")

    print("\nMechanics confirmed. peft_model.save_pretrained('csm-lora') to keep the adapter.")
    print("Restart the runtime before returning to the inference cells above.")
else:
    print("Skipped. Set RUN_FINETUNE_DEMO = True to run it.")

## Notes, limits, and where this differs from Sesame

**What's faithful to the paper**

- Interleaved text+audio conditioning — the context buffer keeps waveforms for both speakers
- Speaker-id tokens (`[0]` / `[1]`) so the model tracks whose voice is whose
- The two-transformer split: backbone → codebook 0 (semantic), depth decoder → codebooks 1–31 (acoustic)
- Mimi at 24 kHz / 12.5 Hz, with the frame budget computed against the real 2048-position window
- Compute amortization (`depth_decoder_labels_ratio = 1/16`) in the fine-tuning cell

---

## Where these projects actually fail

Ranked by how often each one sinks the build, rather than by how interesting it is to work on — and
notice that the top three are *not* architecture problems:

| # | Failure | Status here |
|---|---|---|
| 1 | **Voice consistency / fine-tuning data.** The checkpoint has no fixed voice; context conditioning supplies one | ✅ pinned anchor, never evicted · fine-tuning cell for the real fix |
| 2 | **End-of-speech tuning.** Biggest contributor to "it feels slow", and the cheapest to fix | ✅ client-side VAD instead of a fixed timer |
| 3 | **LLM output register.** Teams spend weeks on latency while the assistant talks in bullet points | ✅ speech-first system prompt |
| 4 | **The `generation_id` race.** Cheap to fix, embarrassing in production | ⛔ N/A — see below |
| 5 | **Text normalization.** Every mispronounced price is invisible to the developer who wrote it | ✅ full normalization pass |

## What a real deployment needs that a notebook cannot have

A notebook is **half duplex by construction**: `record → transcribe → think → speak → play` runs to
completion, and playback is a finished base64 blob handed to an `<audio>` element. There is no
persistent socket, no streaming playback, and therefore nothing to interrupt. Everything below is
deliberately *not* implemented here, and would be the next build:

- **One persistent `wss://` connection** per session, multiplexed by message type — one TLS+TCP
  handshake paid at session start rather than per turn (200–400 ms on mobile you never spend again).
  The cost is head-of-line blocking, so cap audio frames at 20–40 ms of PCM and never send a whole
  LLM response in one frame.
- **The `generation_id` invariant.** The user interrupts at packet 60; the server cancels; packets
  61–68 are already in flight carrying no evidence they're obsolete, so the client plays them. Fix:
  increment `generation_id` **synchronously, before any await**, every producer re-checks before
  every send — and the client raises its own filter locally on barge-in rather than waiting for
  permission, because a server round trip is 40–150 ms and anything past ~100 ms of continued speech
  reads as being ignored.
- **Fade, don't cut.** Stopping at an arbitrary sample is a broadband transient. A 15 ms linear ramp
  is inaudible and removes it. (`fade_edges()` here already does this at chunk seams.)
- **Two asyncio traps.** `task.cancel()` raises at the next `await` — CSM's generation loop is tight
  and GPU-bound, so a naive implementation has no next await and barge-in latency silently becomes
  however long the sentence was. Run it in a thread executor with a `threading.Event` checked per
  frame. And `cancel()` is a *request*: await the task afterwards and swallow the error, or you leak
  GPU memory and leave CSM holding a partial RVQ codebook sequence. Discard the partial output, don't
  resume from it.
- **A 60–120 ms jitter buffer**, which sets your interruption floor — anything in the buffer is audio
  you've already committed to and must throw away.
- **Instrument the stale-packet counter.** If `droppedStale` is always zero, your filter has never
  been exercised and you haven't tested barge-in — you've tested that nothing arrives late on
  localhost.

Two framing points from the guide worth keeping in view: **build the transport layer against a
boring commercial streaming TTS first**, and swap CSM in only once it passes listening tests on its
own — debugging an unsolved transport problem through an unsolved model problem is how six-week
projects become six-month projects. And **latency and expressiveness are separate problems**: CSM's
advantage is prosody, not speed, and its time-to-first-audio is roughly comparable to a cascaded
pipeline once you add ASR and LLM time. "Use CSM" is not a latency strategy.

**What's different, honestly**

| | Sesame's demo | This notebook |
|---|---|---|
| Voice | Fine-tuned CSM variant | Base CSM-1B + a pinned anchor |
| Size | Up to Medium (8B / 300M) | Tiny (1B / 100M) — the only open one |
| Turn-taking | Full duplex, can be interrupted | Half duplex, fixed-length recording |
| Latency | Sub-second, streaming | ~1–3 s to first audio on a T4 |
| Watermark | `silentcipher` on output | **None** — see below |

The duplex gap isn't laziness on our part; it's the open problem Sesame themselves flag as future
work. CSM decides *how* to speak, never *when*. Real turn-taking needs a model of conversational
dynamics that hasn't been released.

**Tuning cheatsheet**

| Symptom | Fix |
|---|---|
| Voice changes between turns | Anchor got evicted or is too short — check `VOICE_ANCHOR.cost()`, reroll `VOICE_SEED` |
| Flat, robotic delivery | Context too thin. Let the conversation run longer; keep audio in the turns |
| Static / garbled audio | You're on fp16 with a T4 — set `CSM_USE_FP16_ON_T4 = False` |
| Rambling, unspeakable replies | Tighten `SYSTEM_PROMPT`; lower `max_new_tokens` |
| Slurring or drift on long sentences | Lower `hard_max` in `SpeechChunker` |
| Click or pop between sentences | Raise the `fade_edges` ramp from 10 ms to ~15 ms |
| Numbers/prices read wrong | Extend `SPOKEN_AS_WORD` / `_UNITS`, or check `DATE_ORDER` |
| Acronym spelled that shouldn't be | Add it to `SPOKEN_AS_WORD` — no rule can infer this |
| RTF above 1.0× (stutter risk) | `COMPILE_CSM = True`, or shorter replies |
| VAD cuts you off mid-sentence | Raise `VAD_SILENCE_MS` |
| VAD never stops recording | Raise `VAD_THRESHOLD` — the room is louder than the gate |
| Turns get slower as you talk | Lower `MAX_CONTEXT_TURNS`; every segment costs prefill |
| OOM | `LLM_BACKEND = "api"` (frees 3 GB), or `ASR_MODEL_ID = "openai/whisper-base"` |
| Too slow | `whisper-base`, shorter replies, or a paid L4/A100 runtime (which also unlocks bf16) |
| `400 Developer instruction is not enabled` | A system prompt reached the API — Gemma won't take one. Fold it into a user turn |
| `429 RESOURCE_EXHAUSTED` | Free tier is ~15 req/min. The retry handles bursts; slow down or upgrade the key |
| `PERMISSION_DENIED` / empty model list | Key is wrong or restricted — regenerate at [aistudio.google.com/apikey](https://aistudio.google.com/apikey) |
| Mic does nothing | Use desktop Chrome/Edge, allow the permission, or type via Step 13 |
| Weights re-download every session | Mount Drive and point `HF_HOME` at it |

**What else you could move off the GPU.** Only Whisper, and probably shouldn't — it's 1.6 GB and
sub-second locally, whereas uploading audio per turn adds latency on the critical path. The one case
for it: Gemini (not Gemma) models accept audio natively, so a single API call could replace *both*
Whisper and Gemma. That's a real simplification, but it means dropping Gemma for Gemini — your call.
**CSM cannot move.** Nobody hosts it, which is the whole reason this notebook needs a GPU at all.

**Responsible use** — Sesame's license prohibits impersonating real individuals without consent,
deception and misinformation, and illegal or harmful use. Two things worth taking seriously:

1. **Watermarking is on** (Step 7c). The 🤗 Transformers integration doesn't apply one, so this
   notebook adds `silentcipher` back explicitly. Leave it on: it's what makes synthetic speech
   identifiable, it's an explicit condition of the model's terms, and transparency rules are moving
   toward requiring exactly this. If you want the performance back, `COMPILE_CSM = True` — don't buy
   it by stripping the watermark. **The bundled key is public**; generate your own before deploying.
2. **Consent.** The "use my own voice" cell is voice cloning. Record yourself, or someone who has
   actually agreed — not a clip off the internet.

**Sources**

- [Crossing the uncanny valley of conversational voice — Sesame](https://www.sesame.com/research/crossing_the_uncanny_valley_of_voice)
- [SesameAILabs/csm on GitHub](https://github.com/SesameAILabs/csm)
- [`sesame/csm-1b` model card](https://huggingface.co/sesame/csm-1b)
- [CSM in 🤗 Transformers](https://huggingface.co/docs/transformers/model_doc/csm)
- [Mimi codec (Kyutai)](https://huggingface.co/kyutai/mimi)
- [Gemma 3](https://huggingface.co/blog/gemma3) · [Run Gemma with the Gemini API](https://ai.google.dev/gemma/docs/core/gemma_on_gemini_api) · [google-genai SDK](https://github.com/googleapis/python-genai)
- [Get an API key — Google AI Studio](https://aistudio.google.com/apikey)